In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

from lightgbm import LGBMClassifier


# ============================================================
# 【读取步骤】
# 读取最终清洗后的训练集
# ============================================================

model_train = pd.read_parquet(
    "data/processed/model_train_clean.parquet"
)


# ============================================================
# 【准备步骤】
# 分离特征和目标变量
# ============================================================

y = (
    model_train["TARGET"]
    .astype("int8")
    .copy()
)

X = (
    model_train
    .drop(
        columns=[
            "TARGET",
            "SK_ID_CURR"
        ]
    )
    .copy()
)


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 【识别字段】
# 区分类别特征和数值特征
# ============================================================

categorical_columns = (
    X_train
    .select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

numeric_columns = (
    X_train
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)


print(
    "类别特征数量:",
    len(categorical_columns)
)

print(
    "数值特征数量:",
    len(numeric_columns)
)

类别特征数量: 24
数值特征数量: 415


In [2]:
# ============================================================
# 【数据预处理】
# 类别字段编码，数值字段保持原值
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier


lightgbm_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_columns
        ),
        (
            "categorical",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            categorical_columns
        )
    ],
    remainder="drop"
)

In [3]:
# ============================================================
# 【模型建立】
# 建立 LightGBM 分类模型
# ============================================================

lightgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [4]:
# ============================================================
# 【模型流水线】
# 合并预处理和 LightGBM 模型
# ============================================================

lightgbm_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            lightgbm_preprocessor
        ),
        (
            "model",
            lightgbm_model
        )
    ]
)

In [5]:
# ============================================================
# 【模型训练】
# 训练 LightGBM
# ============================================================

lightgbm_pipeline.fit(
    X_train,
    y_train
)

print(
    "LightGBM 训练完成"
)

LightGBM 训练完成


In [6]:
# ============================================================
# 【模型预测】
# 生成验证集违约概率
# ============================================================

lgb_valid_probabilities = (
    lightgbm_pipeline
    .predict_proba(X_valid)[:, 1]
)


lgb_valid_predictions = (
    lgb_valid_probabilities >= 0.5
).astype("int8")


print(
    "预测概率最小值:",
    lgb_valid_probabilities.min()
)

print(
    "预测概率最大值:",
    lgb_valid_probabilities.max()
)

print(
    "预测为违约的客户比例:",
    lgb_valid_predictions.mean()
)

预测概率最小值: 0.011494505809419447
预测概率最大值: 0.962282029778367
预测为违约的客户比例: 0.2924572785067395


In [7]:
# ============================================================
# 【模型评估】
# 计算 ROC-AUC、PR-AUC 和分类指标
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)


lgb_roc_auc = roc_auc_score(
    y_valid,
    lgb_valid_probabilities
)

lgb_pr_auc = average_precision_score(
    y_valid,
    lgb_valid_probabilities
)


print(
    "LightGBM Validation ROC-AUC:",
    round(lgb_roc_auc, 4)
)

print(
    "LightGBM Validation PR-AUC:",
    round(lgb_pr_auc, 4)
)


print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_valid,
        lgb_valid_predictions,
        digits=4
    )
)


print(
    "Confusion Matrix:"
)

print(
    confusion_matrix(
        y_valid,
        lgb_valid_predictions
    )
)

LightGBM Validation ROC-AUC: 0.7889
LightGBM Validation PR-AUC: 0.2929

Classification Report:
              precision    recall  f1-score   support

           0     0.9651    0.7428    0.8395     56538
           1     0.1916    0.6941    0.3003      4965

    accuracy                         0.7389     61503
   macro avg     0.5783    0.7184    0.5699     61503
weighted avg     0.9026    0.7389    0.7960     61503

Confusion Matrix:
[[41997 14541]
 [ 1519  3446]]


In [8]:
# ============================================================
# 【模型对比】
# 对比 Logistic Regression、Random Forest 和 LightGBM
# ============================================================

model_comparison = pd.DataFrame(
    {
        "model": [
            "Logistic Regression",
            "Random Forest",
            "LightGBM"
        ],
        "roc_auc": [
            0.7803,
            0.7609,
            lgb_roc_auc
        ],
        "pr_auc": [
            0.2701,
            0.2337,
            lgb_pr_auc
        ]
    }
)


model_comparison.sort_values(
    "roc_auc",
    ascending=False
).round(4)

,model,roc_auc,pr_auc
2,LightGBM,0.7889,0.2929
0,Logistic Regression,0.7803,0.2701
1,Random Forest,0.7609,0.2337


In [9]:
# ============================================================
# 【阈值分析】
# 比较 LightGBM 在不同阈值下的模型表现
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)


# 1. 设置候选阈值
thresholds = np.arange(
    0.10,
    0.81,
    0.05
)


# 2. 保存每个阈值的评估结果
lgb_threshold_results = []

for threshold in thresholds:

    threshold_predictions = (
        lgb_valid_probabilities >= threshold
    ).astype("int8")

    lgb_threshold_results.append(
        {
            "threshold": threshold,

            "precision": precision_score(
                y_valid,
                threshold_predictions,
                zero_division=0
            ),

            "recall": recall_score(
                y_valid,
                threshold_predictions,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_valid,
                threshold_predictions,
                zero_division=0
            ),

            "accuracy": accuracy_score(
                y_valid,
                threshold_predictions
            ),

            "predicted_default_rate": (
                threshold_predictions.mean()
            )
        }
    )


# 3. 转换成 DataFrame
lgb_threshold_comparison = pd.DataFrame(
    lgb_threshold_results
)


# 4. 查看结果
lgb_threshold_comparison.round(4)

,threshold,precision,recall,f1_score,accuracy,predicted_default_rate
0,0.10,0.0864,0.9919,0.1590,0.1529,0.9265
1,0.15,0.0946,0.9742,0.1725,0.2456,0.8310
2,0.20,0.1050,0.9535,0.1892,0.3403,0.7329
3,0.25,0.1161,0.9202,0.2062,0.4282,0.6397
4,0.30,0.1284,0.8826,0.2241,0.5067,0.5551
5,0.35,0.1412,0.8383,0.2417,0.5754,0.4792
6,0.40,0.1563,0.7954,0.2613,0.6369,0.4108
7,0.45,0.1726,0.7462,0.2804,0.6907,0.3490
8,0.50,0.1916,0.6941,0.3003,0.7389,0.2925
9,0.55,0.2131,0.6278,0.3181,0.7828,0.2379


In [10]:
# ============================================================
# 【检查步骤】
# 找出 LightGBM F1最高的阈值
# ============================================================

lgb_best_f1_result = (
    lgb_threshold_comparison
    .sort_values(
        "f1_score",
        ascending=False
    )
    .head(1)
)

lgb_best_f1_result.round(4)

,threshold,precision,recall,f1_score,accuracy,predicted_default_rate
11,0.65,0.2697,0.4858,0.3468,0.8523,0.1454


In [11]:
# ============================================================
# 【业务阈值检查】
# Recall至少达到70%的候选阈值
# ============================================================

lgb_recall_70_results = (
    lgb_threshold_comparison[
        lgb_threshold_comparison["recall"] >= 0.70
    ]
    .sort_values(
        "precision",
        ascending=False
    )
)

lgb_recall_70_results.round(4)

,threshold,precision,recall,f1_score,accuracy,predicted_default_rate
7,0.45,0.1726,0.7462,0.2804,0.6907,0.3490
6,0.40,0.1563,0.7954,0.2613,0.6369,0.4108
5,0.35,0.1412,0.8383,0.2417,0.5754,0.4792
4,0.30,0.1284,0.8826,0.2241,0.5067,0.5551
3,0.25,0.1161,0.9202,0.2062,0.4282,0.6397
2,0.20,0.1050,0.9535,0.1892,0.3403,0.7329
1,0.15,0.0946,0.9742,0.1725,0.2456,0.8310
0,0.10,0.0864,0.9919,0.1590,0.1529,0.9265


In [12]:
# ============================================================
# 【保存结果】
# 保存 LightGBM 阈值分析与模型对比结果
# ============================================================

lgb_threshold_comparison.to_csv(
    "data/processed/lightgbm_threshold_results.csv",
    index=False
)

model_comparison.to_csv(
    "data/processed/final_model_comparison.csv",
    index=False
)

print(
    "LightGBM阈值结果和模型对比结果保存完成"
)

LightGBM阈值结果和模型对比结果保存完成
